# Tuned GBDT comparison

Confronto multi-seed tra B-score logit, GNN corrente, decoder antisimmetrico e
GBDT tuned. Gli artefatti devono condividere match, target, split e scope.

In [1]:
import os
from pathlib import Path
import sys

START = Path.cwd().resolve()
PROJECT_ROOT = START if (START / "edge_feature_ablation").is_dir() else START.parent
sys.path.insert(0, str(PROJECT_ROOT / "edge_feature_ablation"))
from multi_seed_summary import multi_seed_comparison

In [2]:
TOURNAMENT_SCOPE = os.environ.get(
    "TENNIS_RUN_SCOPE", "slams_masters"
)  # or "slams"
SEEDS = [42, 123, 456, 789, 2026]
RESULTS_ROOT = PROJECT_ROOT / "results" / "frozen_predictions" 

In [3]:
per_seed, aggregate = multi_seed_comparison(
    RESULTS_ROOT,
    scope=TOURNAMENT_SCOPE,
    seeds=SEEDS,
    experiments=[
        "bscore_logit",
        "current_gnn",
        "antisymmetric_decoder",
        "gbdt_tuned",
    ],
)
aggregate

,accuracy_mean,accuracy_std,log_loss_mean,log_loss_std,brier_mean,brier_std,accuracy_wins_vs_current,log_loss_wins_vs_current,brier_wins_vs_current
experiment,,,,,,,,,
antisymmetric_decoder,0.667109,0.002801,0.609226,0.003558,0.210325,0.001304,3,4,4
bscore_logit,0.644191,0.001824,0.626019,0.000152,0.216032,0.000078,<NA>,<NA>,<NA>
current_gnn,0.667215,0.003702,0.612380,0.002889,0.211303,0.001032,<NA>,<NA>,<NA>
gbdt_tuned,0.669496,0.002712,0.601504,0.002711,0.207433,0.001109,<NA>,<NA>,<NA>


In [4]:
per_seed

,seed,experiment,n_matches,accuracy,log_loss,brier,evaluation_hash
0,42,bscore_logit,3770.0,0.644032,0.626188,0.216104,575e4dc208092a717d3f90fdef049a97b754ce6c3b5c60...
1,42,current_gnn,3770.0,0.667639,0.611911,0.211465,575e4dc208092a717d3f90fdef049a97b754ce6c3b5c60...
2,42,antisymmetric_decoder,3770.0,0.671353,0.604410,0.208496,575e4dc208092a717d3f90fdef049a97b754ce6c3b5c60...
3,42,gbdt_tuned,3770.0,0.668435,0.605318,0.209073,575e4dc208092a717d3f90fdef049a97b754ce6c3b5c60...
4,123,bscore_logit,3770.0,0.644828,0.625838,0.215922,5009472ab692bd145fd64377421eb583bea5c4caea3906...
5,123,current_gnn,3770.0,0.670822,0.609460,0.209919,5009472ab692bd145fd64377421eb583bea5c4caea3906...
6,123,antisymmetric_decoder,3770.0,0.668170,0.607231,0.209720,5009472ab692bd145fd64377421eb583bea5c4caea3906...
7,123,gbdt_tuned,3770.0,0.666313,0.602416,0.207710,5009472ab692bd145fd64377421eb583bea5c4caea3906...
8,456,bscore_logit,3770.0,0.642440,0.626155,0.216103,a540bf7aa0175613d8d95383db2745d52fcab459c9aa38...
9,456,current_gnn,3770.0,0.665517,0.610175,0.210665,a540bf7aa0175613d8d95383db2745d52fcab459c9aa38...


In [5]:
per_seed.pivot(
    index="seed",
    columns="experiment",
    values=["accuracy", "log_loss", "brier"],
)

accuracy                                      \
experiment antisymmetric_decoder bscore_logit current_gnn gbdt_tuned   
seed                                                                   
42                      0.671353     0.644032    0.667639   0.668435   
123                     0.668170     0.644828    0.670822   0.666313   
456                     0.666578     0.642440    0.665517   0.668170   
789                     0.665252     0.642706    0.661804   0.672944   
2026                    0.664191     0.646950    0.670292   0.671618   

                        log_loss                                      \
experiment antisymmetric_decoder bscore_logit current_gnn gbdt_tuned   
seed                                                                   
42                      0.604410     0.626188    0.611911   0.605318   
123                     0.607231     0.625838    0.609460   0.602416   
456                     0.611695     0.626155    0.610175   0.597847   
789                     0.613388     0.625906    0.613743   0.600781   
2026                    0.609406     0.626009    0.616612   0.601161   

                           brier                                      
experiment antisymmetric_decoder bscore_logit current_gnn gbdt_tuned  
seed                                                                  
42                      0.208496     0.216104    0.211465   0.209073  
123                     0.209720     0.215922    0.209919   0.207710  
456                     0.211115     0.216103    0.210665   0.206010  
789                     0.211894     0.215987    0.211932   0.207113  
2026                    0.210402     0.216043    0.212535   0.207259

In [6]:
pivot = per_seed.pivot(
    index="seed",
    columns="experiment",
    values=["accuracy", "log_loss", "brier"],
)
for baseline in ["bscore_logit", "current_gnn", "antisymmetric_decoder"]:
    print("\nGBDT minus", baseline)
    display(pivot.xs("gbdt_tuned", level="experiment", axis=1)
            - pivot.xs(baseline, level="experiment", axis=1))


GBDT minus bscore_logit


,accuracy,log_loss,brier
seed,,,
42,0.024403,-0.020870,-0.007031
123,0.021485,-0.023423,-0.008212
456,0.025729,-0.028308,-0.010093
789,0.030239,-0.025125,-0.008874
2026,0.024668,-0.024848,-0.008784



GBDT minus current_gnn


,accuracy,log_loss,brier
seed,,,
42,0.000796,-0.006593,-0.002392
123,-0.004509,-0.007045,-0.002209
456,0.002653,-0.012328,-0.004655
789,0.011141,-0.012962,-0.004819
2026,0.001326,-0.015451,-0.005276



GBDT minus antisymmetric_decoder


,accuracy,log_loss,brier
seed,,,
42,-0.002918,0.000908,0.000576
123,-0.001857,-0.004815,-0.002010
456,0.001592,-0.013848,-0.005104
789,0.007692,-0.012607,-0.004780
2026,0.007427,-0.008245,-0.003143
